# Pipeline Runtime Profiling (small candidate subset)

This notebook profiles high-load pipeline modules on a small, configurable subset of candidates.

Targeted stages:
- **Events scoring path** (`malca.events.process_lightcurve`)
- **Pre-periodicity validation path** (`malca.periodicity_gate._evaluate_periodicity_worker`)

It includes cProfile and wall-time profiling, per-module timing breakdowns, and a final summary section.


## 1) Environment setup


In [1]:
import io
import cProfile
import pstats
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

candidate_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
repo_root = next(
    (p for p in candidate_roots if (p / "malca").is_dir() and (p / "malca" / "events.py").exists()),
    Path.cwd(),
)

for path in (repo_root, repo_root / "malca"):
    sp = str(path.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)

from malca.notebook_paths import (
    discover_bundled_lightcurve_paths,
    infer_run_dir,
    localize_lightcurve_frame_paths,
    resolve_repo_path,
)

print(f"Repo root: {repo_root}")
print(f"Notebook path: {Path.cwd()}")


Repo root: /home/calder/code/malca
Notebook path: /home/calder/code/malca/malca/notebooks


## 2) Configuration (edit as needed)


In [2]:
# Optional explicit events/filter table path (parquet/csv).
# Example: repo_root / "output" / "runs" / "<run_id>" / "results" / "lc_events_results_13_13.5.parquet"
RESULTS_TABLE_PATH = None
RUN_DIR = None

# Auto-discovery patterns used when RESULTS_TABLE_PATH is None
RESULTS_GLOBS = [
    "**/lc_events_results*.parquet",
    "**/lc_events_filtered*.parquet",
    "**/lc_events_results*.csv",
    "**/lc_events_filtered*.csv",
]

SUBSET_SIZE = 6
RANDOM_SEED = 42
PROFILE_SORT = "cumtime"
PROFILE_LIMIT = 30

# Event-scoring path kwargs
EVENT_STAGE_KWARGS = {
    "trigger_mode": "posterior_prob",
    "logbf_threshold_dip": 5.0,
    "logbf_threshold_jump": 5.0,
    "significance_threshold": 99.99997,
    "p_points": 80,
    "p_min_dip": None,
    "p_max_dip": None,
    "p_min_jump": None,
    "p_max_jump": None,
    "mag_points": 12,
    "run_min_points": 2,
    "max_gap_points": 1,
    "run_max_gap_days": None,
    "run_min_duration_days": None,
    "baseline_tag": "gp",
    "compute_event_prob": True,
    "auto_filter_bad_cameras": True,
}

# Pre-periodicity worker kwargs (smaller n_periods for quick subset profiling).
# The current pregate is CE-only, so the worker tuple mirrors
# malca.periodicity_gate._evaluate_periodicity_worker directly.
PREGATE_STAGE_KWARGS = {
    "n_periods": 3000,
}


## 3) Load candidate subset

The notebook prefers an events/filter results table. If unavailable, it falls back to direct light-curve file discovery.

If both are empty, set `RESULTS_TABLE_PATH` manually.


In [3]:
def _read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    return pd.read_csv(path)


def _discover_results_table(root: Path) -> Path | None:
    search_roots = [root / "output", root / "results", root]
    for search_root in search_roots:
        if not search_root.exists():
            continue
        for pattern in RESULTS_GLOBS:
            matches = sorted(search_root.glob(pattern))
            if matches:
                return matches[-1]
    return None


def _resolve_run_dir(run_dir_value, table_path: Path | None) -> Path | None:
    candidates = []
    resolved_run_dir = resolve_repo_path(run_dir_value, repo_root=repo_root) if run_dir_value else None
    if resolved_run_dir is not None:
        candidates.append(resolved_run_dir)
    candidates.extend([
        table_path,
        repo_root / "output" / "runs" / "latest",
        repo_root / "output" / "runs",
    ])
    for candidate in candidates:
        run_dir = infer_run_dir(candidate)
        if run_dir is not None:
            return run_dir

    runs_root = repo_root / "output" / "runs"
    if runs_root.is_dir():
        run_dirs = sorted(p for p in runs_root.iterdir() if p.is_dir())
        if run_dirs:
            return run_dirs[-1]
    return None


def _discover_fallback_paths(run_dir: Path | None) -> list[Path]:
    if run_dir is not None:
        paths = discover_bundled_lightcurve_paths(run_dir, limit=SUBSET_SIZE)
        if paths:
            return paths

    runs_root = repo_root / "output" / "runs"
    if runs_root.is_dir():
        return discover_bundled_lightcurve_paths(runs_root, limit=SUBSET_SIZE)
    return []


def _coerce_excluded_cameras(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    return s if s else None


selected_table = resolve_repo_path(RESULTS_TABLE_PATH, repo_root=repo_root) if RESULTS_TABLE_PATH else _discover_results_table(repo_root)
selected_run_dir = _resolve_run_dir(RUN_DIR, selected_table)
subset_df = pd.DataFrame(columns=["path", "excluded_cameras", "source"])

if selected_table is not None and selected_table.exists():
    df_all = _read_table(selected_table)
    print(f"Loaded table: {selected_table}")
    print(f"Rows: {len(df_all):,}; columns: {len(df_all.columns):,}")
    if selected_run_dir is not None:
        print(f"Resolving light curves against bundled assets in {selected_run_dir}")

    path_col = "dat_path" if "dat_path" in df_all.columns else ("path" if "path" in df_all.columns else None)
    if path_col is None:
        print("No path/dat_path column found in table; using bundled fallback file discovery.")
    else:
        df_work = df_all.copy()
        df_work, localized_counts = localize_lightcurve_frame_paths(
            df_work,
            run_dir=selected_run_dir,
            repo_root=repo_root,
            path_columns=("dat_path", "path", "lc_path"),
        )
        if localized_counts:
            print(f"Localized stored light-curve paths: {localized_counts}")

        if "failed_any" in df_work.columns:
            failed = df_work["failed_any"]
            if pd.api.types.is_bool_dtype(failed):
                keep_mask = ~failed.fillna(False)
            elif pd.api.types.is_numeric_dtype(failed):
                keep_mask = failed.fillna(0).astype(float) == 0.0
            else:
                keep_mask = ~failed.fillna("").astype(str).str.lower().isin({"1", "true", "t", "yes", "y"})
            df_work = df_work[keep_mask].copy()

        sig_cols = [c for c in ("dip_significant", "jump_significant") if c in df_work.columns]
        if sig_cols:
            sig_mask = pd.Series(False, index=df_work.index)
            for col in sig_cols:
                sig_mask = sig_mask | df_work[col].fillna(False).astype(bool)
            if sig_mask.any():
                df_work = df_work[sig_mask].copy()

        rank_components = []
        for col in ("dip_bayes_factor", "jump_bayes_factor", "periodicity_score", "pre_periodicity_score"):
            if col in df_work.columns:
                rank_components.append(pd.to_numeric(df_work[col], errors="coerce"))
        if rank_components:
            df_work["_rank"] = pd.concat(rank_components, axis=1).max(axis=1, skipna=True)
            df_work = df_work.sort_values("_rank", ascending=False, na_position="last")

        exists_mask = df_work[path_col].map(lambda x: Path(str(x)).expanduser().exists() if pd.notna(x) else False)
        if not bool(exists_mask.all()):
            print(f"Retained {int(exists_mask.sum()):,} rows with bundled local light curves.")
        df_work = df_work[exists_mask].copy()

        keep_cols = [path_col]
        if "excluded_cameras" in df_work.columns:
            keep_cols.append("excluded_cameras")
        subset_df = df_work[keep_cols].copy().head(SUBSET_SIZE)
        subset_df = subset_df.rename(columns={path_col: "path"})
        if "excluded_cameras" not in subset_df.columns:
            subset_df["excluded_cameras"] = None
        subset_df["excluded_cameras"] = subset_df["excluded_cameras"].map(_coerce_excluded_cameras)
        subset_df["path"] = subset_df["path"].astype(str)
        subset_df["source"] = str(selected_table)

if subset_df.empty:
    fallback_paths = _discover_fallback_paths(selected_run_dir)
    if fallback_paths:
        subset_df = pd.DataFrame({
            "path": [str(p) for p in fallback_paths[:SUBSET_SIZE]],
            "excluded_cameras": [None] * min(SUBSET_SIZE, len(fallback_paths)),
            "source": ["bundle_assets_fallback"] * min(SUBSET_SIZE, len(fallback_paths)),
        })
        print(f"Using bundled fallback light-curve paths (n={len(subset_df)}).")
    else:
        print("No candidate rows or bundled fallback files found. Set RESULTS_TABLE_PATH or RUN_DIR and re-run this section.")

if not subset_df.empty:
    subset_df = subset_df.reset_index(drop=True)
    print(f"Profiling subset size: {len(subset_df)}")
    display(subset_df.head(10))


Loaded table: /home/calder/code/malca/output/runs/runs_march18_bundle_all/results/lc_events_results_all.parquet
Rows: 20,990; columns: 78
Resolving light curves against bundled assets in /home/calder/code/malca/output/runs/runs_march18_bundle_all
Localized stored light-curve paths: {'path': 9569}
Retained 9,569 rows with bundled local light curves.
Profiling subset size: 6


,path,excluded_cameras,source
0,/home/calder/code/malca/output/runs/runs_march...,None,/home/calder/code/malca/output/runs/runs_march...
1,/home/calder/code/malca/output/runs/runs_march...,None,/home/calder/code/malca/output/runs/runs_march...
2,/home/calder/code/malca/output/runs/runs_march...,None,/home/calder/code/malca/output/runs/runs_march...
3,/home/calder/code/malca/output/runs/runs_march...,None,/home/calder/code/malca/output/runs/runs_march...
4,/home/calder/code/malca/output/runs/runs_march...,None,/home/calder/code/malca/output/runs/runs_march...
5,/home/calder/code/malca/output/runs/runs_march...,None,/home/calder/code/malca/output/runs/runs_march...


## 4) Profiling helpers (cProfile + wall time + module breakdowns)


In [4]:
def pstats_to_frames(stats_obj: pstats.Stats) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    for (filename, lineno, funcname), (cc, nc, tt, ct, callers) in stats_obj.stats.items():
        file_path = Path(filename)
        rows.append({
            "file": str(filename),
            "module": file_path.stem,
            "func": funcname,
            "lineno": int(lineno),
            "primitive_calls": int(cc),
            "total_calls": int(nc),
            "tottime_s": float(tt),
            "cumtime_s": float(ct),
        })

    funcs_df = pd.DataFrame(rows)
    if funcs_df.empty:
        return funcs_df, pd.DataFrame(columns=["module", "cumtime_s", "tottime_s", "func_count"])

    funcs_df = funcs_df.sort_values("cumtime_s", ascending=False).reset_index(drop=True)
    module_df = (
        funcs_df.groupby("module", as_index=False)
        .agg(cumtime_s=("cumtime_s", "sum"), tottime_s=("tottime_s", "sum"), func_count=("func", "count"))
        .sort_values("cumtime_s", ascending=False)
        .reset_index(drop=True)
    )
    return funcs_df, module_df


def profile_callable(func, *args, sort_by: str = PROFILE_SORT, stats_limit: int = PROFILE_LIMIT, **kwargs):
    profiler = cProfile.Profile()
    t0 = time.perf_counter()
    result = profiler.runcall(func, *args, **kwargs)
    elapsed = time.perf_counter() - t0

    buf = io.StringIO()
    stats_obj = pstats.Stats(profiler, stream=buf).strip_dirs().sort_stats(sort_by)
    stats_obj.print_stats(stats_limit)
    stats_text = buf.getvalue()

    funcs_df, modules_df = pstats_to_frames(stats_obj)
    return result, elapsed, stats_text, funcs_df, modules_df


def batch_time_stage(stage_name: str, records: list[dict], runner):
    rows = []
    for record in records:
        path = str(record.get("path"))
        excluded = record.get("excluded_cameras")
        t0 = time.perf_counter()
        ok = True
        error = None
        try:
            _ = runner(path, excluded)
        except Exception as exc:
            ok = False
            error = str(exc)
        elapsed = time.perf_counter() - t0
        rows.append({
            "stage": stage_name,
            "path": path,
            "excluded_cameras": excluded,
            "elapsed_s": elapsed,
            "ok": ok,
            "error": error,
        })
    return pd.DataFrame(rows)


## 5) Stage runners (events scoring + pre-periodicity path)

Imports are wrapped so missing optional dependencies are surfaced with clear messages.


In [5]:
IMPORT_ERRORS = {}

try:
    import malca.events as events
except Exception as exc:
    events = None
    IMPORT_ERRORS["events"] = str(exc)

try:
    import malca.periodicity_gate as pregate
except Exception as exc:
    pregate = None
    IMPORT_ERRORS["periodicity_gate"] = str(exc)

if IMPORT_ERRORS:
    print("Some modules could not be imported:")
    for name, msg in IMPORT_ERRORS.items():
        print(f"  - {name}: {msg}")
    print("Install missing deps, then re-run this cell.")


def run_events_stage(path: str, excluded_cameras: str | None = None):
    if events is None:
        raise RuntimeError(f"malca.events import failed: {IMPORT_ERRORS.get('events')}")

    kwargs = dict(EVENT_STAGE_KWARGS)
    return events.process_lightcurve(
        str(path),
        excluded_cameras=excluded_cameras,
        **kwargs,
    )


def build_pregate_worker_args(path: str, excluded_cameras: str | None = None):
    if pregate is None:
        raise RuntimeError(f"malca.periodicity_gate import failed: {IMPORT_ERRORS.get('periodicity_gate')}")

    checkpoint_key = pregate._checkpoint_key(str(path), excluded_cameras)

    return (
        str(path),
        checkpoint_key,
        excluded_cameras,
        float(getattr(pregate, "BAD_CAMERA_SCATTER_RATIO_THRESHOLD", 2.5)),
        float(getattr(pregate, "CLEAN_LC_MAX_ERROR_ABSOLUTE", 0.2)),
        float(getattr(pregate, "CLEAN_LC_MAX_ERROR_SIGMA", 5.0)),
        float(getattr(pregate, "PRE_PERIODICITY_MIN_PERIOD", 0.2)),
        float(getattr(pregate, "PRE_PERIODICITY_MAX_PERIOD", 100.0)),
        int(PREGATE_STAGE_KWARGS.get("n_periods", getattr(pregate, "PRE_PERIODICITY_N_PERIODS", 3000))),
        float(getattr(pregate, "PRE_PERIODICITY_CE_SNR_THRESHOLD", 10.0)),
        int(getattr(pregate, "PRE_PERIODICITY_MIN_POINTS", 50)),
        float(getattr(pregate, "PRE_PERIODICITY_SCATTER_RATIO_MAX", 0.9)),
    )


def run_pregate_stage(path: str, excluded_cameras: str | None = None):
    if pregate is None:
        raise RuntimeError(f"malca.periodicity_gate import failed: {IMPORT_ERRORS.get('periodicity_gate')}")
    return pregate._evaluate_periodicity_worker(build_pregate_worker_args(path, excluded_cameras))


/home/calder/miniforge3/envs/malca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 6) cProfile on one representative candidate per stage


In [6]:
events_profile_modules = pd.DataFrame()
pregate_profile_modules = pd.DataFrame()

if subset_df.empty:
    print("Subset is empty. Configure RESULTS_TABLE_PATH and re-run above cells.")
elif IMPORT_ERRORS:
    print("Resolve import errors shown earlier before profiling stages.")
else:
    sample = subset_df.iloc[0].to_dict()
    sample_path = str(sample["path"])
    sample_excluded = sample.get("excluded_cameras")

    print(f"Profiling sample path: {sample_path}")

    events_result, events_elapsed, events_stats_text, events_funcs_df, events_profile_modules = profile_callable(
        run_events_stage, sample_path, sample_excluded
    )
    print(f"\n[events] elapsed: {events_elapsed:.3f} s")
    print(events_stats_text)
    display(events_profile_modules.head(15))

    pregate_result, pregate_elapsed, pregate_stats_text, pregate_funcs_df, pregate_profile_modules = profile_callable(
        run_pregate_stage, sample_path, sample_excluded
    )
    print(f"\n[pre_periodicity_gate] elapsed: {pregate_elapsed:.3f} s")
    print(pregate_stats_text)
    display(pregate_profile_modules.head(15))


Profiling sample path: /home/calder/code/malca/output/runs/runs_march18_bundle_all/bundle_assets/lightcurves/463857054014.dat3

[events] elapsed: 3.198 s
         2296149 function calls (2261758 primitive calls) in 3.197 seconds

   Ordered by: cumulative time
   List reduced from 2903 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    3.198    3.198 306211832.py:22(run_events_stage)
        1    0.000    0.000    3.198    3.198 events.py:1130(process_lightcurve)
        1    0.000    0.000    1.837    1.837 events.py:1065(score_lightcurve)
        2    0.001    0.001    1.794    0.897 events.py:646(score_events_bayesian)
      221    0.008    0.000    1.760    0.008 _minpack_py.py:585(curve_fit)
       35    0.002    0.000    1.425    0.041 events.py:178(classify_run_morphology)
       24    0.001    0.000    1.230    0.051 least_squares.py:241(least_squares)
       24    0.000    0.000    1.214    0.05

,module,cumtime_s,tottime_s,func_count
0,events,8.310523,0.048557,23
1,_minpack_py,3.708796,0.193172,7
2,306211832,3.197847,0.000245,1
3,utils,2.634066,0.630630,26
4,trf,2.485984,0.088469,3
5,least_squares,1.847479,0.010803,10
6,~,1.351734,0.471867,240
7,_numdiff,1.309825,0.177222,8
8,frame,0.906175,0.038072,27
9,common,0.874235,0.261435,80



[pre_periodicity_gate] elapsed: 1.146 s
         1156090 function calls (1128773 primitive calls) in 1.146 seconds

   Ordered by: cumulative time
   List reduced from 1343 to 30 due to restriction <30>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    1.146    1.146 306211832.py:56(run_pregate_stage)
        1    0.000    0.000    1.146    1.146 periodicity_gate.py:361(_evaluate_periodicity_worker)
        1    0.000    0.000    1.009    1.009 lightcurve_io.py:63(load_lightcurve_df)
        1    0.000    0.000    0.991    0.991 utils.py:905(filter_bad_cameras)
        1    0.011    0.011    0.555    0.555 utils.py:705(identify_offset_cameras)
     3420    0.040    0.000    0.448    0.000 frame.py:4062(__getitem__)
        1    0.015    0.015    0.395    0.395 utils.py:821(identify_catastrophic_outlier_cameras)
     1033    0.009    0.000    0.331    0.000 frame.py:4130(_getitem_bool_array)
     1264    0.008    0.000    0.262    0

,module,cumtime_s,tottime_s,func_count
0,utils,2.101593,0.063007,21
1,periodicity_gate,1.190806,0.002173,19
2,306211832,1.145802,0.000092,2
3,lightcurve_io,1.008997,0.000111,1
4,frame,0.946720,0.065149,28
5,generic,0.754357,0.110723,61
6,managers,0.674167,0.051100,61
7,series,0.640854,0.062354,39
8,common,0.365984,0.053920,60
9,periodogram,0.312443,0.109483,6


## 7) Batch wall-time profiling on the subset


In [7]:
timing_df = pd.DataFrame()
stage_summary = pd.DataFrame()

if subset_df.empty:
    print("Subset is empty; skipping batch timing.")
elif IMPORT_ERRORS:
    print("Resolve import errors shown earlier before running batch timing.")
else:
    records = subset_df.to_dict(orient="records")

    df_events_t = batch_time_stage("events_scoring", records, run_events_stage)
    df_pregate_t = batch_time_stage("pre_periodicity_gate", records, run_pregate_stage)

    timing_df = pd.concat([df_events_t, df_pregate_t], ignore_index=True)
    display(timing_df)

    stage_summary = (
        timing_df.groupby("stage", as_index=False)
        .agg(
            n=("elapsed_s", "size"),
            ok=("ok", "sum"),
            mean_s=("elapsed_s", "mean"),
            median_s=("elapsed_s", "median"),
            p90_s=("elapsed_s", lambda x: float(np.nanpercentile(x, 90))),
            max_s=("elapsed_s", "max"),
        )
        .sort_values("mean_s", ascending=False)
        .reset_index(drop=True)
    )
    display(stage_summary)


,stage,path,excluded_cameras,elapsed_s,ok,error
0,events_scoring,/home/calder/code/malca/output/runs/runs_march...,None,2.279499,True,None
1,events_scoring,/home/calder/code/malca/output/runs/runs_march...,None,1.699433,True,None
2,events_scoring,/home/calder/code/malca/output/runs/runs_march...,None,0.677781,True,None
3,events_scoring,/home/calder/code/malca/output/runs/runs_march...,None,3.238788,True,None
4,events_scoring,/home/calder/code/malca/output/runs/runs_march...,None,2.066079,True,None
5,events_scoring,/home/calder/code/malca/output/runs/runs_march...,None,2.126828,True,None
6,pre_periodicity_gate,/home/calder/code/malca/output/runs/runs_march...,None,0.778610,True,None
7,pre_periodicity_gate,/home/calder/code/malca/output/runs/runs_march...,None,0.838457,True,None
8,pre_periodicity_gate,/home/calder/code/malca/output/runs/runs_march...,None,0.226431,True,None
9,pre_periodicity_gate,/home/calder/code/malca/output/runs/runs_march...,None,0.631427,True,None


,stage,n,ok,mean_s,median_s,p90_s,max_s
0,events_scoring,6,6,2.014735,2.096453,2.759144,3.238788
1,pre_periodicity_gate,6,6,0.650335,0.713544,0.808533,0.838457


## 8) Summary

This final cell reports top bottlenecks and practical next steps.


In [8]:
if subset_df.empty:
    print("No profiling summary available yet: subset is empty.")
    print("Set RESULTS_TABLE_PATH to a valid events/filter table and re-run.")
else:
    print(f"Subset size profiled: {len(subset_df)}")

    if not stage_summary.empty:
        print("\nStage timing summary (seconds):")
        display(stage_summary)
    else:
        print("No stage timing data collected.")

    if not events_profile_modules.empty:
        print("\nTop modules by cumulative time (events scoring sample):")
        display(events_profile_modules.head(10))

    if not pregate_profile_modules.empty:
        print("\nTop modules by cumulative time (pre-periodicity sample):")
        display(pregate_profile_modules.head(10))

    print("\nInterpretation notes:")
    print("- Prioritize modules with highest cumulative time for optimization.")
    print("- Compare events vs pre-periodicity medians to identify dominant stage on this subset.")
    print("- Increase SUBSET_SIZE gradually after validating behavior on this quick profile pass.")


Subset size profiled: 6

Stage timing summary (seconds):


,stage,n,ok,mean_s,median_s,p90_s,max_s
0,events_scoring,6,6,2.014735,2.096453,2.759144,3.238788
1,pre_periodicity_gate,6,6,0.650335,0.713544,0.808533,0.838457



Top modules by cumulative time (events scoring sample):


,module,cumtime_s,tottime_s,func_count
0,events,8.310523,0.048557,23
1,_minpack_py,3.708796,0.193172,7
2,306211832,3.197847,0.000245,1
3,utils,2.634066,0.630630,26
4,trf,2.485984,0.088469,3
5,least_squares,1.847479,0.010803,10
6,~,1.351734,0.471867,240
7,_numdiff,1.309825,0.177222,8
8,frame,0.906175,0.038072,27
9,common,0.874235,0.261435,80



Top modules by cumulative time (pre-periodicity sample):


,module,cumtime_s,tottime_s,func_count
0,utils,2.101593,0.063007,21
1,periodicity_gate,1.190806,0.002173,19
2,306211832,1.145802,0.000092,2
3,lightcurve_io,1.008997,0.000111,1
4,frame,0.946720,0.065149,28
5,generic,0.754357,0.110723,61
6,managers,0.674167,0.051100,61
7,series,0.640854,0.062354,39
8,common,0.365984,0.053920,60
9,periodogram,0.312443,0.109483,6



Interpretation notes:
- Prioritize modules with highest cumulative time for optimization.
- Compare events vs pre-periodicity medians to identify dominant stage on this subset.
- Increase SUBSET_SIZE gradually after validating behavior on this quick profile pass.
